# Laboratório 2 - Visão Computacional

## Parte 2: SIFT

(A) Elaborar um programa OpenCV, que realize a operação “Feature Matching + Homography to find Objects”, deste [tutorial](https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html) - este programa deverá ler duas imagens previamente gravadas que contenham o mesmo objeto em posições distintas. No final mostrar conforme o tutorial, as correspondências obtidas.

In [ ]:
%pip install opencv-python
%pip install numpy
%pip install matplotlib

Carregamos as dependências necessárias:

In [ ]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

Lemos as imagens:

In [ ]:
img1 = cv.imread(r"images/IMG_3401.jpeg", cv.IMREAD_GRAYSCALE)
img2 = cv.imread(r"images/IMG_3402.jpeg", cv.IMREAD_GRAYSCALE)

In [ ]:
MIN_MATCH_COUNT = 10

# Initiate SIFT detector
sift = cv.SIFT_create()

# find the keypoints and descriptors with SIFT
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
search_params = dict(checks = 50)

flann = cv.FlannBasedMatcher(index_params, search_params)

matches = flann.knnMatch(des1,des2,k=2)

# store all the good matches as per Lowe's ratio test.
good = []
for m,n in matches:
    if m.distance < 0.7*n.distance:
        good.append(m)

In [ ]:
if len(good)>MIN_MATCH_COUNT:
    src_pts = np.float32([ kp1[m.queryIdx].pt for m in good ]).reshape(-1,1,2)
    dst_pts = np.float32([ kp2[m.trainIdx].pt for m in good ]).reshape(-1,1,2)

    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC,5.0)
    matchesMask = mask.ravel().tolist()

    h,w = img1.shape
    pts = np.float32([ [0,0],[0,h-1],[w-1,h-1],[w-1,0] ]).reshape(-1,1,2)
    dst = cv.perspectiveTransform(pts,M)

    img2 = cv.polylines(img2,[np.int32(dst)],True,255,3, cv.LINE_AA)

else:
    print("Not enough matches are found - {}/{}".format(len(good), MIN_MATCH_COUNT))
    matchesMask = None

In [ ]:
draw_params = dict(matchColor = (0,255,0), # draw matches in green color
                   singlePointColor = None,
                   matchesMask = matchesMask, # draw only inliers
                   flags = 2)

img3 = cv.drawMatches(img1,kp1,img2,kp2,good,None,**draw_params)

plt.imshow(img3, 'gray'),plt.show()

(B) Elaborar outro programa OpenCV, modificando o codigo acima, para fazer a leitura de duas webcams pelo mesmo código, executando nessas duas imagens a correspondencia SIFT do item A acima, e mostrando o mesmo tipo de resultado numa janela de video a cada leitura das webcams.

In [ ]:
cap1 = cv.VideoCapture(0)
cap2 = cv.VideoCapture(1)

if not cap1.isOpened():
    print("Cannot open camera 1")
    exit()
    
if not cap2.isOpened():
    print("Cannot open camera 2")
    exit()

MIN_MATCH_COUNT = 10
sift = cv.SIFT_create()

FLANN_INDEX_KDTREE = 1
index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
search_params = dict(checks=50)
flann = cv.FlannBasedMatcher(index_params, search_params)

while True:
    # Capture frame-by-frame
    ret1, frame1 = cap1.read()
    ret2, frame2 = cap2.read()
    
    # if frame is read correctly ret is True
    if not ret1:
        print("Can't receive frame 1 (stream end?). Exiting ...")
        break

    if not ret2:
        print("Can't receive frame 2 (stream end?). Exiting ...")
        break

    gray1 = cv.cvtColor(frame1, cv.COLOR_BGR2GRAY)
    gray2 = cv.cvtColor(frame2, cv.COLOR_BGR2GRAY)

    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)

    if des1 is not None and des2 is not None and len(des1) > 1 and len(des2) > 1:
        matches = flann.knnMatch(des1,des2,k=2)

        # store all the good matches as per Lowe's ratio test.
        good = []
        for m,n in matches:
            if m.distance < 0.7*n.distance:
                good.append(m)
        
        matchesMask = None
        if len(good)>MIN_MATCH_COUNT:
            src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
            dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

            M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
            if M is not None:
                matchesMask = mask.ravel().tolist()

                h,w = gray1.shape
                pts = np.float32([[0, 0], [0, h-1], [w-1, h-1], [w-1, 0]]).reshape(-1, 1, 2)
                dst = cv.perspectiveTransform(pts, M)

                frame2 = cv.polylines(frame2, [np.int32(dst)], True, (0, 0, 255), 3, cv.LINE_AA)
        else:
            cv.putText(frame2, f"Matches: {len(good)}/{MIN_MATCH_COUNT}", (10, 30), 
                       cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        draw_params = dict(matchColor=(0, 255, 0),  
                           singlePointColor=None,
                           matchesMask=matchesMask,  
                           flags=2)

        img3 = cv.drawMatches(frame1, kp1, frame2, kp2, good, None, **draw_params)
        cv.imshow('SIFT Matches', np.vstack((img3, np.hstack((frame1, frame2)))))
    else:
        cv.imshow('SIFT Matches', np.hstack((frame1, frame2)))

    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap1.release()
cap2.release() 
cv.destroyAllWindows()